# Task 4.5 — Geo Mean HAI Across All Variants (D28)

**4.5 Predict antibody breadth — all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Geo mean / Metric: Spearman correlation
* Full description: Geometric mean HAI titer across all measured variants at Day 28

---

## Design notes

**y-values:** `breadth_delta` = log2(geomean HAI D28) − log2(geomean HAI D0), computed from the 17 challenge strains (16/17 have D28 data in training; `H3N2 A/Massachusetts/18/2022` is missing). Only participants with ≥ `MIN_STRAINS` strains at both D0 and D28 are included. Since Spearman only cares about ranking, the delta scale is fine for evaluation.

**Output scale:** Challenge CSV uses `np.exp2(geomean_d0_challenge + predicted_breadth_delta)` — the raw geometric mean HAI titer at D28.

**Spearman correlation:** ranks predictions and truth; rewards monotonic agreement regardless of scale. Robust to outliers. Score: 1.0 = perfect, 0.0 = no signal, -1.0 = reversed.

**5-fold cross-validation:** each participant's prediction is made by a model that never saw them during training.

In [ ]:
TARGET_COL = 'breadth_delta'
AUTOML_MAX_MODELS = 10
AUTOML_SEED = 1
AUTO_ML_MAX_RUNTIME_SECONDS = 60 * 1
ONLY_TRANSCRIPTOMICS_PARTICIPANTS = False
FORCE_KEEP_TRANSCRIPTOMICS = True
MISSING_THRESHOLD = 0.8
MIN_STRAINS = 5

In [ ]:
CSV_PATH = '../cleaned_data/train_combined.csv'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = 'submission'

In [ ]:
import io
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

In [ ]:
challenge_data = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_combined.csv')
print(f'Challenge shape: {challenge_data.shape}')

---
## Preprocessing

Each step prints shape after filtering. The same coercions applied to training data are applied to challenge data in the same cell.

### Preprocessing — define challenge strains and compute target

Define the 17 challenge strains, compute log2-geomeans at D0 and D28 for each training participant,
and set `breadth_delta = geomean_d28 − geomean_d0`. Challenge data gets the same `geomean_d0`
computation — it is needed later to reconstruct the raw geomean HAI at D28 for the submission.

In [ ]:
CHALLENGE_STRAINS = [
    'H1N1 A/California/7/2009',
    'H1N1 A/Brisbane/2/2018',
    'H1N1 A/Guangdong-Maonan/SWL1536/2019',
    'H1N1 A/Victoria/2570/2019',
    'H1N1 A/Victoria/4897/2022',
    'H3N2 A/Hong Kong/4801/2014',
    'H3N2 A/Singapore/INFIMH-160019/2016',
    'H3N2 A/Kansas/14/2017',
    'H3N2 A/Hong Kong/2671/2019',
    'H3N2 A/South Australia/34/2019',
    'H3N2 A/Tasmania/503/2020',
    'H3N2 A/Darwin/9/2021',
    'H3N2 A/Massachusetts/18/2022',
    'Vic B/Colorado/6/2017',
    'Vic B/Washington/2/2019',
    'Vic B/Austria/1359417/2021',
    'Yam B/Phuket/3073/2013',
]

df = pd.read_csv(CSV_PATH)
print(f'Pre-filtered shape: {df.shape}')

d0_cols = [f'HAI_{s}_d0' for s in CHALLENGE_STRAINS if f'HAI_{s}_d0' in df.columns]
d28_cols = [f'HAI_{s}_d28' for s in CHALLENGE_STRAINS if f'HAI_{s}_d28' in df.columns]
print(f'Challenge strains with D0 data: {len(d0_cols)} / {len(CHALLENGE_STRAINS)}')
print(f'Challenge strains with D28 data: {len(d28_cols)} / {len(CHALLENGE_STRAINS)}')

df['geomean_d0'] = df[d0_cols].mean(axis=1, skipna=True)
df['geomean_d28'] = df[d28_cols].mean(axis=1, skipna=True)
df['n_strains_d0'] = df[d0_cols].notna().sum(axis=1)
df['n_strains_d28'] = df[d28_cols].notna().sum(axis=1)
df[TARGET_COL] = df['geomean_d28'] - df['geomean_d0']

# Challenge: same geomean_d0 needed to reconstruct raw D28 titer at predict time
d0_cols_chal = [f'HAI_{s}_d0' for s in CHALLENGE_STRAINS if f'HAI_{s}_d0' in challenge_data.columns]
challenge_data['geomean_d0'] = challenge_data[d0_cols_chal].mean(axis=1, skipna=True)
print(f'Challenge geomean_d0 computed from {len(d0_cols_chal)} strains.')

### Filter by minimum strain coverage

Keep only participants with ≥ `MIN_STRAINS` strains measured at both D0 and D28 so the geomean is
based on enough data points to be meaningful.

In [ ]:
df = df[
    (df['n_strains_d0'] >= MIN_STRAINS) &
    (df['n_strains_d28'] >= MIN_STRAINS) &
    df[TARGET_COL].notna()
].reset_index(drop=True)
print(f'After MIN_STRAINS={MIN_STRAINS} and target-notna filter: {df.shape}')

### Drop all-null columns

In [ ]:
all_null_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=all_null_cols)
print(f'Dropped {len(all_null_cols)} all-null columns. New shape: {df.shape}')

### Coerce HAI string columns to numeric

Some `HAI_*` titer columns are stored as strings (e.g. censored values like `<10` or stringified NaN).
H2O treats those as categorical at training time but numeric at prediction time, which crashes `predict()`.
We coerce all `HAI_*` string columns to numeric (non-numeric values become NaN). Genuine categoricals
(demographics) are left alone.

In [ ]:
# str_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
# print(f'String columns ({len(str_cols)}):', str_cols)

# hai_str_cols = [c for c in str_cols if c.startswith('HAI_')]
# for c in hai_str_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')
# print(f'\nCoerced {len(hai_str_cols)} HAI string columns to numeric.')
# print(f'Remaining string columns: {df.select_dtypes(include=["object", "str"]).columns.tolist()}')

# # Apply same coercion to challenge data so type mismatch can't crash predict()
# challenge_hai_str_cols = [c for c in challenge_data.select_dtypes(include=['object', 'str']).columns
#                           if c.startswith('HAI_')]
# for c in challenge_hai_str_cols:
#     challenge_data[c] = pd.to_numeric(challenge_data[c], errors='coerce')
# print(f'Coerced {len(challenge_hai_str_cols)} HAI string columns in challenge data.')

### Drop constant features

Any column with only one unique value (numeric or categorical) carries no signal. Since we filtered
to participants with this target, some categorical features may collapse to a single value here.

In [ ]:
constant_cols = [c for c in df.columns
                 if c != TARGET_COL and df[c].nunique(dropna=True) <= 1]
df = df.drop(columns=constant_cols)
print(f'Dropped {len(constant_cols)} constant columns: {constant_cols}')
print(f'Shape: {df.shape}')

### Transcriptomics stats

In [ ]:
tran_cols = [c for c in df.columns if c.startswith('TRAN_')]
df_tran = df[df[tran_cols].notna().any(axis=1)].reset_index(drop=True)
print(f'Participants with TRAN data: {df_tran.shape[0]} / {df.shape[0]} '
      f'({df_tran.shape[0] / df.shape[0]:.1%})')

if ONLY_TRANSCRIPTOMICS_PARTICIPANTS:
    df = df_tran
    print(f'Filtered df to TRAN-only participants. Shape: {df.shape}')

In [ ]:
num_to_show = 15
tran_corrs = df_tran[tran_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
top = tran_corrs.loc[tran_corrs.abs().sort_values(ascending=False).head(num_to_show).index]
top = top.sort_values()

vmax = top.abs().max()
colors = plt.cm.coolwarm((top.values / vmax + 1) / 2)

fig, ax = plt.subplots(figsize=(7, max(4, num_to_show * 0.3)))
ax.barh(top.index, top.values, color=colors, edgecolor='black', linewidth=0.3)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(f'Pearson r with {TARGET_COL}')
ax.set_title(f'Top {num_to_show}/{len(tran_cols)} Transcriptomics PCs by correlation with target')
ax.set_xlim(-vmax * 1.15, vmax * 1.15)
plt.tight_layout()
plt.show()

### Drop sparse features

Drop columns missing in more than `MISSING_THRESHOLD` of rows — too gappy for tree models to learn
from and slow training enough that only GLMs finish.

In [ ]:
miss_frac = df.isna().mean()
sparse_cols = miss_frac[miss_frac > MISSING_THRESHOLD].index

if FORCE_KEEP_TRANSCRIPTOMICS:
    cols_to_drop = [c for c in sparse_cols if c != TARGET_COL and c not in tran_cols]
else:
    cols_to_drop = [c for c in sparse_cols if c != TARGET_COL]

df = df.drop(columns=cols_to_drop)
print(f'Dropped {len(cols_to_drop)} columns with >{MISSING_THRESHOLD:.0%} missing.')
print(f'Shape: {df.shape}')

In [ ]:
surviving_tran = [c for c in df.columns if c.startswith('TRAN_')]
print(f'TRAN columns surviving sparse filter: {len(surviving_tran)} / {len(tran_cols)}')

### Drop other-task target columns

The challenge participants only have baseline data (demographics + d0 + d7) — d28 and d365
measurements don't exist for them yet. Also drop the constructed `geomean_d28` and `n_strains_*`
columns which are unavailable at prediction time.

In [ ]:
leakage_cols = [c for c in df.columns
                if (c.endswith('_d28') and c != TARGET_COL) or c.endswith('_d365')]
leakage_cols += [c for c in ('geomean_d28', 'n_strains_d0', 'n_strains_d28') if c in df.columns]
leakage_cols = list(dict.fromkeys(leakage_cols))
df = df.drop(columns=leakage_cols)
print(f'Dropped {len(leakage_cols)} leakage columns.')
print(f'Final shape: {df.shape}')

In [ ]:
# Force float64 on all remaining HAI feature columns to prevent H2O from mistyping
# sparsely-populated columns as categorical, which crashes predict() on the challenge set.
hai_feature_cols = [c for c in df.columns if c.startswith('HAI_') and c != TARGET_COL]
for c in hai_feature_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    if c in challenge_data.columns:
        challenge_data[c] = pd.to_numeric(challenge_data[c], errors='coerce')
print(f'Forced float64 on {len(hai_feature_cols)} HAI feature columns in training and challenge data.')

In [ ]:
print(f'Final shape: {df.shape}')
print(f'Target ({TARGET_COL}) stats (log2 delta):\n{df[TARGET_COL].describe()}')
print(f'\ndtype counts:\n{df.dtypes.value_counts()}')

---
## AutoML Setup

In [ ]:
warnings.filterwarnings('ignore', category=UserWarning, module='h2o')
h2o.init()

In [ ]:
hai_col_types = {c: 'real' for c in hai_feature_cols}
tran_col_types = {c: 'real' for c in surviving_tran}
col_types = {**hai_col_types, **tran_col_types}
data = h2o.H2OFrame(df, column_types=col_types)
print(f'H2OFrame shape: {data.shape}')

---
## AutoML Training

In [ ]:
y = TARGET_COL
x = [c for c in data.columns if c not in (y, 'participant_id')]
print(f'Training samples: {data.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(
    max_models=AUTOML_MAX_MODELS,
    seed=AUTOML_SEED,
    nfolds=5,
    keep_cross_validation_predictions=True,
    max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS,
)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=data)
print('Training complete.')

In [ ]:
lb_df = aml.leaderboard.as_data_frame(use_multi_thread=True)
lb_df

In [ ]:
# Best overall (may be a StackedEnsemble) — used for predictions and saved as the deliverable
top_model = h2o.get_model(lb_df['model_id'].iloc[0])

# Best non-ensemble — used for Spearman CV and varimp, since StackedEnsembles don't expose
# cross-validation holdout predictions in the same way base models do.
top_base_model_id = lb_df[~lb_df['model_id'].str.contains('StackedEnsemble')]['model_id'].iloc[0]
top_base_model = h2o.get_model(top_base_model_id)

cv_preds = top_base_model.cross_validation_holdout_predictions().as_data_frame(use_multi_thread=True)['predict']
actuals = data[y].as_data_frame(use_multi_thread=True)[y]

rho, pval = spearmanr(actuals, cv_preds)
print(f'Prediction model:  {top_model.model_id}')
print(f'Scoring model:     {top_base_model.model_id}')
print(f'Task 4.5 — Spearman (best base model 5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Best base model: {top_base_model.model_id}')
varimp = top_base_model.varimp(use_pandas=True)
display(varimp.head(20))
top_base_model.varimp_plot(num_of_features=20)

In [ ]:
challenge_col_types = {c: 'real' for c in hai_feature_cols if c in challenge_data.columns}
challenge_col_types.update({c: 'real' for c in surviving_tran if c in challenge_data.columns})
challenge_hf = h2o.H2OFrame(challenge_data, column_types=challenge_col_types)
y_pred = top_model.predict(challenge_hf).as_data_frame(use_multi_thread=True)['predict']

challenge_geomean_d0 = challenge_data['geomean_d0'].values

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.5': np.exp2(challenge_geomean_d0 + y_pred),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_5.csv', index=False)
results

In [ ]:
model_path = h2o.save_model(model=top_model, path='.', filename='models/model_4.5', force=True)
print(f'Model saved to: {model_path}')

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** TBD (re-run to populate)
- **Scoring model:** TBD (best non-StackedEnsemble base, used for CV Spearman)
- **CV Spearman:** TBD
- **Training samples:** TBD | **Features:** TBD
- **Model saved:** `automl_models/models/model_4.5`

**Target:** `breadth_delta` = log2(geomean HAI D28) − log2(geomean HAI D0) across up to 16/17
challenge strains with D28 data in training. `H3N2 A/Massachusetts/18/2022` has no D28 data and
is excluded from D28 averaging. Only participants with ≥ `MIN_STRAINS` strains at both timepoints
are included as training rows.

**Output:** `np.exp2(geomean_d0_challenge + predicted_breadth_delta)` — raw geometric mean HAI
titer at D28 for each challenge participant.

**Feature set:** demographics + HAI d0/d7 columns + TRAN PCs from `combined.csv`, after dropping
columns >80% missing and all d28/d365 leakage columns. `FORCE_KEEP_TRANSCRIPTOMICS = True`
preserves all TRAN PCs regardless of sparsity. All HAI and TRAN columns are explicitly typed
`real` on both training and challenge H2OFrames to prevent auto-type mismatches.

**Imputation note:** No manual imputation. H2O's tree models handle `NaN` natively at each split,
eliminating leakage that would arise from pre-computing medians on the full training set.

Submission saved to `submission/task_4_5.csv`.

To reload the model: `h2o.load_model('automl_models/models/model_4.5')`